# Cervidae taxonomy tree

This example renders the deer family with photos on species nodes and text-only taxonomic groups. Its data archive is a pinned snapshot of Wikipedia's [List of cervids](https://en.wikipedia.org/wiki/List_of_cervids), with reviewed species photos and range maps. It does not fetch data at notebook runtime.

The source taxonomy is a classification, not a newly inferred phylogeny. See `examples/data/cervidae-tree.zip` for the complete source table, revision metadata, and image licenses.

In [ ]:
from __future__ import annotations

import base64
import html
import json
import mimetypes
import re
import zipfile
from pathlib import Path

import ipywidgets as W
import networkx as nx
import traitlets as T

import ipyelk
from ipyelk.elements import Port, shapes
from ipyelk.loaders import NXLoader
from ipyelk.pipes import flows as F
from ipyelk.tools import ToggleCollapsedTool

In [ ]:
DATA = Path("data/cervidae-tree.zip")

with zipfile.ZipFile(DATA) as archive:
    image_review = json.loads(archive.read("images/review.json"))
    taxonomy = json.loads(archive.read("taxonomy.json"))
    source = json.loads(archive.read("source/list_of_cervids.json"))
    species = json.loads(archive.read("species.json"))
    image_metadata = json.loads(archive.read("images/metadata.json"))
    for item in image_metadata:
        if re.search(
            r"range|distribution|map|habitat",
            item["file_title"],
            re.IGNORECASE,
        ):
            image_review.setdefault(item["archive_name"], {"role": "range"})
    images = {
        image["subject"]: (image["archive_name"], archive.read(image["archive_name"]))
        for image in image_metadata
        if image.get("kind") == "species"
        and image["archive_name"].startswith("images/user/")
        and image_review.get(image["archive_name"], {}).get("role", "photo") == "photo"
    }
    wikipedia_photos = {
        image["subject"]: (image["archive_name"], archive.read(image["archive_name"]))
        for image in image_metadata
        if image.get("kind") == "species"
        and not image["archive_name"].startswith("images/user/")
        and image_review.get(image["archive_name"], {}).get("role", "photo") == "photo"
    }
    range_images = {
        image["subject"]: (image["archive_name"], archive.read(image["archive_name"]))
        for image in sorted(
            image_metadata,
            key=lambda item: "crop" in image_review.get(item["archive_name"], {}),
        )
        if image_review.get(image["archive_name"], {}).get("role", image.get("kind"))
        == "range"
    }
    card_overrides = {
        image["subject"]: (image["archive_name"], archive.read(image["archive_name"]))
        for image in image_metadata
        if image.get("kind") == "card_photo"
        and image_review.get(image["archive_name"], {}).get("role", "photo") == "photo"
    }
    species_images = {**images, **wikipedia_photos, **card_overrides}


print(
    f"{len(species)} species from Wikipedia revision {source['revision_id']} "
    f"({source['timestamp']})"
)

In [ ]:
def data_url(filename: str, content: bytes) -> str:
    decision = image_review.get(filename, {})
    crop = decision.get("crop", [0, 0, 1, 1])
    if crop != [0, 0, 1, 1]:
        x, y, width, height = crop
        iw, ih = decision["dimensions"]
        source_mime = mimetypes.guess_type(filename)[0] or "image/jpeg"
        original = f"data:{source_mime};base64," + base64.b64encode(content).decode()
        content = f'<svg xmlns="http://www.w3.org/2000/svg" width="{width * iw}" height="{height * ih}" viewBox="{x * iw} {y * ih} {width * iw} {height * ih}"><image width="{iw}" height="{ih}" href="{original}"/></svg>'.encode()
        filename = "crop.svg"
    mime_type, _ = mimetypes.guess_type(filename)
    encoded = base64.b64encode(content).decode()
    return f"data:{mime_type or 'application/octet-stream'};base64,{encoded}"


species_by_genus = {}
for record in species:
    species_by_genus.setdefault(record["genus"], []).append(record)
taxonomy_children = {}


def node_attributes(
    name: str, label: str | None = None, *, is_species: bool = False
) -> dict:
    has_image = name in images
    placement = (
        "H_RIGHT V_CENTER OUTSIDE"
        if has_image
        else "H_LEFT V_CENTER INSIDE"
        if is_species
        else "H_CENTER V_CENTER INSIDE"
    )
    width, height = (90, 64) if has_image else (max(132, 9 * len(label or name)), 42)
    attributes = {
        "id": name,
        "width": width,
        "height": height,
        "labels": [
            {
                "text": label or name,
                "layoutOptions": {"org.eclipse.elk.nodeLabels.placement": placement},
            }
        ],
        "ports": [
            Port(
                id=f"{name}.{key}",
                x=x,
                y=height / 2,
                width=0,
                height=0,
                layoutOptions={"org.eclipse.elk.port.side": side},
                properties={"cssClasses": "taxonomy-anchor", "key": f"{name}.{key}"},
            )
            for key, x, side in [("in", 0, "WEST"), ("out", width, "EAST")]
        ],
        "layoutOptions": {
            "org.eclipse.elk.portConstraints": "FIXED_POS",
            "org.eclipse.elk.nodeSize.constraints": "",
        },
    }
    if has_image:
        filename, content = images[name]
        attributes.update(
            properties={
                "shape": shapes.Image(use=data_url(filename, content)).model_dump()
            },
        )
    else:
        attributes["properties"] = {
            "cssClasses": "taxonomy-text",
            "shape": shapes.SVG(
                use=f'<rect width="{width}" height="{height}" fill="transparent" stroke="none"/>'
            ).model_dump(),
        }
    if is_species and not has_image:
        attributes["properties"]["cssClasses"] = "taxonomy-text taxonomy-species"
    return attributes


def add_taxonomy(
    graph: nx.DiGraph,
    branch: dict | list,
    parent: str | None = None,
) -> None:
    entries = (
        branch.items() if isinstance(branch, dict) else ((name, []) for name in branch)
    )
    for name, children in entries:
        graph.add_node(name, **node_attributes(name))
        if parent is not None:
            graph.add_edge(
                parent, name, sourcePort=f"{parent}.out", targetPort=f"{name}.in"
            )
            taxonomy_children.setdefault(parent, []).append(name)
        add_taxonomy(graph, children, name)
        if name in species_by_genus:
            for record in species_by_genus[name]:
                attributes = node_attributes(
                    record["id"], record["common_name"], is_species=True
                )
                graph.add_node(record["id"], **attributes)
                graph.add_edge(
                    name,
                    record["id"],
                    sourcePort=f"{name}.out",
                    targetPort=record["id"] + ".in",
                )
                taxonomy_children.setdefault(name, []).append(record["id"])


graph = nx.DiGraph()
add_taxonomy(graph, taxonomy)


def toggle_shape(collapsed: bool):
    vertical = '<path d="M 12 7 V 17"/>' if collapsed else ""
    return shapes.SVG(
        use=f'<g stroke="#444" stroke-width="1.5"><circle cx="12" cy="12" r="11" fill="white"/><path d="M 7 12 H 17"/>{vertical}</g>'
    )


toggle_ids = {name: f"{name}.__toggle" for name in graph if graph.out_degree(name)}
toggle_parents = {value: key for key, value in toggle_ids.items()}
render_graph = graph.copy()
for parent, identifier in toggle_ids.items():
    attrs = node_attributes(identifier)
    attrs.update(
        width=24,
        height=24,
        labels=[],
        properties={
            "cssClasses": "taxonomy-toggle",
            "shape": toggle_shape(False).model_dump(),
        },
    )
    for port, x in zip(attrs["ports"], (0, 24)):
        port.x, port.y = x, 12
    render_graph.add_node(identifier, **attrs)
    render_graph.add_edge(
        parent, identifier, sourcePort=f"{parent}.out", targetPort=f"{identifier}.in"
    )
    for child in graph.successors(parent):
        render_graph.remove_edge(parent, child)
        render_graph.add_edge(
            identifier, child, sourcePort=f"{identifier}.out", targetPort=f"{child}.in"
        )
loader = NXLoader(
    default_label_opts={
        "org.eclipse.elk.nodeLabels.placement": "H_CENTER V_BOTTOM OUTSIDE"
    }
)
diagram_source = loader.load(render_graph)
for node in diagram_source.value.children:
    if node.id in toggle_parents:
        node.labels = []
diagram_source.value.layoutOptions.update({
    "org.eclipse.elk.algorithm": "org.eclipse.elk.layered",
    "org.eclipse.elk.contentAlignment": "H_CENTER V_CENTER",
    "org.eclipse.elk.layered.nodePlacement.bk.fixedAlignment": "BALANCED",
    "org.eclipse.elk.layered.spacing.nodeNodeBetweenLayers": "48",
})
diagram = ipyelk.Diagram(
    source=diagram_source,
    layout={
        "height": "auto",
        "min_height": "900px",
        "min_width": "0",
        "flex": "1 1 0",
        "align_self": "stretch",
    },
)
diagram.style = {
    " .taxonomy-text.elknode": {"fill": "transparent", "stroke": "transparent"},
    " .taxonomy-text .elklabel": {"font-weight": "600"},
    " .elkport": {"fill": "transparent", "stroke": "transparent", "opacity": "0"},
}


def descendants(identifier: str) -> tuple[str, ...]:
    direct_children = taxonomy_children.get(identifier, [])
    return tuple(
        child
        for direct_child in direct_children
        for child in (direct_child, *descendants(direct_child))
    )


class TaxonomyCollapseTool(ToggleCollapsedTool):
    taxonomy_descendants = T.Dict()

    def get_related(self, element):
        element_id = element.get_id()
        index = self.selection.get_index()
        return [
            index.from_id(identifier)
            for identifier in self.taxonomy_descendants.get(element_id, ())
        ]


collapse = TaxonomyCollapseTool(
    selection=diagram.view.selection,
    taxonomy_descendants={name: descendants(name) for name in taxonomy_children},
)
diagram.tools = tuple(
    tool for tool in diagram.tools if not isinstance(tool, ToggleCollapsedTool)
)
for tool, icon, tooltip in (
    (diagram.view.fit_tool, "expand", "Fit the tree in the viewport"),
    (diagram.view.center_tool, "crosshairs", "Center the tree in the viewport"),
):
    tool.description = ""
    tool.ui.icon = icon
    tool.ui.tooltip = tooltip
    tool.ui.layout = W.Layout(
        width="30px", height="30px", min_width="30px", padding="0"
    )

In [ ]:
details = W.VBox(
    [
        W.HTML(
            "<section class='cervidae-card cervidae-empty'><h3>Select a taxon</h3><p>Select a species to inspect the frozen Wikipedia snapshot.</p></section>"
        )
    ],
    layout={"width": "410px", "flex": "0 0 410px", "padding": "0"},
)
details.add_class("cervidae-details")
records_by_id = {record["id"]: record for record in species}


def species_details(record: dict) -> str:
    image = species_images.get(record["id"])
    photo = (
        ""
        if image is None
        else (
            f"<img class='cervidae-photo' src='{data_url(*image)}' alt='{html.escape(record['common_name'])}'>"
        )
    )
    facts = [
        ("Range", record["range"]),
        ("Habitat", record["habitat"]),
        ("Diet", record["diet"]),
        ("Size", record["size"]),
    ]
    range_image = range_images.get(record["id"])
    range_html = (
        ""
        if range_image is None
        else f"<img class='cervidae-range-map' src='{data_url(*range_image)}' alt='Geographic range map'>"
    )
    rows = "".join(
        f"<div><dt>{html.escape(label)}</dt><dd>{html.escape(value) or '—'}</dd></div>".replace(
            "</dd>", (range_html if label == "Range" else "") + "</dd>"
        )
        for label, value in facts
    )
    population = record["population"].strip()
    population_text = (
        f"Reported population: {population}"
        if population and population.casefold() != "unknown"
        else "Population not reported in this snapshot."
    )
    status = record["status"]
    status_names = {
        "LC": "Least Concern",
        "NT": "Near Threatened",
        "VU": "Vulnerable",
        "EN": "Endangered",
        "CR": "Critically Endangered",
        "EW": "Extinct in the Wild",
        "EX": "Extinct",
        "DD": "Data Deficient",
        "NE": "Not Evaluated",
    }
    tooltip = html.escape(
        f"{status_names.get(status, status)}. {population_text}", quote=True
    )
    return f"""<section class='cervidae-card'>{photo}<div class='cervidae-heading'>
      <p class='cervidae-kicker'>{html.escape(record["scientific_name"])}</p>
      <div class='cervidae-title-row'><h3>{html.escape(record["common_name"])}</h3>
      <span class='cervidae-status status-{html.escape(status)}' tabindex='0' title='{tooltip}' aria-label='{tooltip}'>{html.escape(status)}</span></div></div>
      <dl class='cervidae-facts' style='padding:0 20px;margin:8px 0 12px'>{rows}</dl></section>"""


def refresh_visibility() -> None:
    index = diagram.view.selection.get_index()
    for name in graph:
        related = list(nx.descendants(graph, name))
        if related:
            control = index.from_id(toggle_ids[name])
            control.properties.hidden = index.from_id(name).properties.hidden
            control.properties.shape = toggle_shape(
                all(index.from_id(child).properties.hidden for child in related)
            )
    for _, edge in index.elements.edges():
        source, target = edge.points()
        edge.properties.hidden = bool(
            source.properties.hidden or target.properties.hidden
        )
    inlet = diagram.pipe.inlet
    inlet.flow = tuple(dict.fromkeys((*inlet.flow, F.Node.hidden)))
    diagram.refresh()


def toggle_taxon(identifier: str) -> None:
    index = diagram.view.selection.get_index()
    related = [index.from_id(name) for name in nx.descendants(graph, identifier)]
    hide = not all(node.properties.hidden for node in related)
    for node in related:
        node.properties.hidden = hide
    refresh_visibility()


def select_taxon(identifier: str) -> None:
    index = diagram.view.selection.get_index()
    for name in nx.ancestors(graph, identifier) | {identifier}:
        index.from_id(name).properties.hidden = False
    diagram.view.selection.ids = (identifier,)
    refresh_visibility()


def group_details(identifier: str):
    children = list(graph.successors(identifier))
    species_count = sum(
        name in records_by_id for name in nx.descendants(graph, identifier)
    )
    heading = W.HTML(
        f"<div class='cervidae-heading'><h3>{html.escape(identifier)}</h3><p>{len(children)} immediate children · {species_count} species in this group</p></div>",
        layout={"flex": "1 1 0", "min_width": "0"},
    )
    index = diagram.view.selection.get_index()
    related = [index.from_id(name) for name in nx.descendants(graph, identifier)]
    toggle = W.Button(
        layout={
            "width": "30px",
            "height": "30px",
            "flex": "0 0 30px",
            "margin": "12px 20px 0 0",
        }
    )

    def refresh_toggle():
        collapsed = all(node.properties.hidden for node in related)
        toggle.description = "+" if collapsed else "-"
        toggle.tooltip = (
            f"{'Expand' if collapsed else 'Collapse'} descendants of {identifier}"
        )
        toggle.disabled = not related

    def toggle_children(button):
        toggle_taxon(identifier)
        refresh_toggle()

    toggle.on_click(toggle_children)
    refresh_toggle()
    rows = [
        W.HBox([heading, toggle], layout={"width": "100%", "align_items": "flex-start"})
    ]
    for child in children:
        record = records_by_id.get(child)
        name = record["common_name"] if record else child
        link = W.Button(
            description=name,
            tooltip=f"Select {name} in the tree",
            layout={"width": "100%", "height": "auto", "margin": "0"},
        )
        link.add_class("cervidae-child-link")
        link.on_click(lambda button, child=child: select_taxon(child))
        content = [link]
        if record:
            content.append(
                W.HTML(
                    f"<em class='cervidae-child-scientific'>{html.escape(record['scientific_name'])}</em>"
                )
            )
        else:
            count = sum(name in records_by_id for name in nx.descendants(graph, child))
            content.append(
                W.HTML(
                    f"<span class='cervidae-child-scientific'>{count} species</span>"
                )
            )
        entry = W.VBox(content, layout={"flex": "1 1 0", "min_width": "0"})
        if record:
            badge = re.search(
                r"<span class='cervidae-status.*?</span>", species_details(record)
            ).group()
            row = W.HBox([W.HTML(badge), entry], layout={"align_items": "center"})
        else:
            row = W.HBox([entry])
        row.add_class("cervidae-child-row")
        rows.append(row)
    card = W.VBox(rows)
    card.add_class("cervidae-card")
    return card


def update_details(change) -> None:
    selected = change["new"]
    if not selected:
        return
    identifier = selected[0]
    if identifier in toggle_parents:
        parent = toggle_parents[identifier]
        toggle_taxon(parent)
        diagram.view.selection.ids = (parent,)
        return
    if identifier in records_by_id:
        details.children = (W.HTML(species_details(records_by_id[identifier])),)
    else:
        details.children = (group_details(identifier),)


diagram.view.selection.observe(update_details, "ids")
chrome = W.HTML("""<style>
.cervidae-details { background: var(--jp-layout-color1); }
.taxonomy-toggle { cursor: pointer; }
.cervidae-child-row { box-sizing: border-box; width: calc(100% - 40px); min-width: 0; flex-shrink: 0; margin: 0 20px; padding: .45rem 0; gap: .6rem; border-top: 1px solid var(--jp-border-color2); }
.cervidae-child-row .widget-html, .cervidae-child-row .jupyter-widget-html { margin: 0; min-width: 0; max-width: 100%; }
.cervidae-child-row .widget-html-content, .cervidae-child-row .jupyter-widget-html-content { min-width: 0; max-width: 100%; overflow-wrap: anywhere; }
.cervidae-child-row:last-child { margin-bottom: 12px; }
.cervidae-child-link.jupyter-button { box-sizing: border-box; min-width: 0; max-width: 100%; overflow-wrap: anywhere; text-align: left; white-space: normal; line-height: 1.4; padding: 0; border: 0; box-shadow: none; background: transparent; color: var(--jp-content-link-color); cursor: pointer; }
.cervidae-child-link.jupyter-button:hover { text-decoration: underline; }
.cervidae-child-link.jupyter-button:focus-visible { outline: 2px solid var(--jp-brand-color1); }
.cervidae-child-scientific { color: var(--jp-ui-font-color2); font-size: .85em; }
.cervidae-card { box-sizing: border-box; width: calc(100% - 24px); margin: 12px; overflow: hidden; border: 1px solid var(--jp-border-color2); border-radius: 12px; background: var(--jp-layout-color1); box-shadow: 0 4px 16px rgba(0,0,0,.12); }
.cervidae-empty { padding: 1.25rem; color: var(--jp-ui-font-color2); }
.cervidae-photo { width: 100%; height: auto; max-height: 320px; display: block; object-fit: contain; background: #fff; }
.cervidae-heading { padding: .7rem 20px .2rem; }
.cervidae-title-row { display: flex; align-items: center; gap: .6rem; }
.cervidae-heading h3 { flex: 1; min-width: 0; margin: 0; font-size: 1.2rem; }
.cervidae-heading p { margin: .45rem 0; color: var(--jp-ui-font-color2); }
.cervidae-range-map { display: block; width: 100%; height: auto; object-fit: contain; margin-top: .5rem; }
.cervidae-kicker { margin: 0; font-style: italic; }
.cervidae-status { display: inline-block; flex: 0 0 auto; cursor: help; padding: .2rem .55rem; border-radius: 999px; font-weight: 700; background: var(--jp-warn-color3); color: var(--jp-ui-inverse-font-color1); }
.status-LC { background: var(--jp-success-color1); } .status-NT { background: var(--jp-warn-color1); } .status-VU, .status-EN, .status-CR { background: var(--jp-error-color1); }
.cervidae-facts { margin: .9rem 0 1.25rem; padding: 0 1.25rem; }
.cervidae-facts div { display: grid; grid-template-columns: 92px minmax(0, 1fr); gap: .75rem; align-items: start; padding: .65rem 0; border-top: 1px solid var(--jp-border-color2); }
.cervidae-facts dt { grid-column: 1; margin: 0; font-size: .72rem; font-weight: 700; letter-spacing: .04em; text-transform: uppercase; color: var(--jp-ui-font-color2); }
.cervidae-facts dd { grid-column: 2; min-width: 0; margin: 0; line-height: 1.42; }
</style>""")
W.VBox([
    chrome,
    W.HBox([diagram, details], layout={"align_items": "stretch", "width": "100%"}),
])

## Attribution

The archive's `images/metadata.json` retains Commons attribution for downloaded images and identifies user-supplied species photos separately. Licenses for the user-supplied photos have not been provided: verify redistribution rights before publishing those assets. `images/user/import.json` records unmatched files; the frozen taxonomy is unchanged.

## Interaction

Select a species to open its frozen Wikipedia detail card. Click the circular +/− junction to the right of a parent to expand/collapse its descendants. The junction and its incoming connection remain visible while collapsed. Child links and the +/− control in group cards work too.